# Student Performance & At-Risk Prediction
## 01: Exploratory Data Analysis & Model Experiments
**Algions AI Labs Private Limited | AI & Machine Learning**

This notebook covers the reproducible exploratory data analysis, data quality checks, feature engineering, and baseline model comparisons for predicting whether a student is likely to score below 40/100 in their next academic assessment.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Plot style config
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

RAW_DATA_PATH = Path("../data/raw/student_assessments.csv")
print("Libraries loaded successfully.")

## 1. Dataset Loading and Quality Inspection
We inspect dataset dimensions, data types, statistical distributions, missing entries, and check for duplicate records.

In [ ]:
df = pd.read_csv(RAW_DATA_PATH)
print(f"Shape: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()

In [ ]:
print("--- Data Types and Missing Values ---")
df.info()

In [ ]:
print("--- Summary Statistics ---")
df.describe().round(2)

In [ ]:
# Missing values and duplicate checks
print(f"Total Missing Values: {df.isnull().sum().sum()}")
print(f"Total Duplicate Records: {df.duplicated().sum()}")
print(f"Number of Unique Students: {df['student_id'].nunique()}")
print(f"Chronological Assessment Periods: {sorted(df['assessment_period'].unique())}")

## 2. Target Distribution: At-Risk Class Balance
Target definition: `at_risk = 1` if `next_assessment_score < 40`, else `0`.

In [ ]:
counts = df["at_risk"].value_counts()
rates = df["at_risk"].value_counts(normalize=True)
print(counts)
print(rates.round(3))

fig, ax = plt.subplots(figsize=(6, 4))
sns.countplot(data=df, x="at_risk", palette=["seagreen", "crimson"], ax=ax)
ax.set_title("At-Risk Class Distribution (0 = Not At Risk, 1 = At Risk)", fontweight="bold")
ax.set_xticklabels(["Not At Risk (0)", "At Risk (1)"])
plt.show()

## 3. Exploratory Data Visualizations
Visualizing distributions for attendance percentage, assignment completion, previous scores, and performance trend.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

sns.histplot(df["attendance_percentage"], kde=True, color="steelblue", ax=axes[0, 0])
axes[0, 0].set_title("Attendance Percentage Distribution")

sns.histplot(df["assignment_completion_percentage"], kde=True, color="darkorange", ax=axes[0, 1])
axes[0, 1].set_title("Assignment Completion Percentage Distribution")

sns.histplot(df["previous_assessment_score"], kde=True, color="purple", ax=axes[1, 0])
axes[1, 0].set_title("Previous Assessment Score Distribution")

sns.histplot(df["performance_trend"], kde=True, color="teal", ax=axes[1, 1])
axes[1, 1].set_title("Performance Trend (Recent Score - Earlier Score)")

plt.tight_layout()
plt.show()

In [ ]:
# Feature distribution by At-Risk class
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(data=df, x="at_risk", y="attendance_percentage", palette=["seagreen", "crimson"], ax=axes[0])
axes[0].set_title("Attendance by At-Risk Status")
axes[0].set_xticklabels(["Not At Risk", "At Risk"])

sns.boxplot(data=df, x="at_risk", y="previous_assessment_score", palette=["seagreen", "crimson"], ax=axes[1])
axes[1].set_title("Previous Score by At-Risk Status")
axes[1].set_xticklabels(["Not At Risk", "At Risk"])

plt.tight_layout()
plt.show()

## 4. Correlation Heatmap
Evaluating collinearity and linear associations with next assessment score and at-risk labels.

In [ ]:
corr_cols = [
    "attendance_percentage", "assignment_completion_percentage",
    "previous_assessment_score", "earlier_assessment_score",
    "missed_assignments", "average_previous_score",
    "performance_trend", "next_assessment_score", "at_risk"
]
corr = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax)
ax.set_title("Correlation Heatmap", fontweight="bold")
plt.show()

## 5. Leakage Prevention and Chronological Splitting
Strict separation: periods 1-3 train, period 4 validation, period 5 test.
Excluding `next_assessment_score`, `at_risk`, and `student_id` from model inputs.

In [ ]:
train_df = df[df["assessment_period"].isin([1, 2, 3])].copy()
val_df = df[df["assessment_period"] == 4].copy()
test_df = df[df["assessment_period"] == 5].copy()

FEATURE_COLS = [
    "assessment_period", "attendance_percentage", "assignment_completion_percentage",
    "previous_assessment_score", "earlier_assessment_score", "missed_assignments",
    "average_previous_score", "performance_trend"
]
TARGET = "at_risk"

X_train, y_train = train_df[FEATURE_COLS], train_df[TARGET]
X_val, y_val = val_df[FEATURE_COLS], val_df[TARGET]
X_test, y_test = test_df[FEATURE_COLS], test_df[TARGET]

print(f"Training set:   {X_train.shape[0]} rows")
print(f"Validation set: {X_val.shape[0]} rows")
print(f"Test set:       {X_test.shape[0]} rows")
print(f"Data leakage check passed: {'next_assessment_score' not in FEATURE_COLS}")

## 6. Baseline & Candidate Model Comparisons
Training and comparing Logistic Regression, Random Forest, and XGBoost on the validation set.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb
from sklearn.metrics import classification_report, f1_score, recall_score, precision_score, roc_auc_score

models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(class_weight="balanced", random_state=42, max_iter=1000))
    ]),
    "Random Forest": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", RandomForestClassifier(n_estimators=200, max_depth=8, min_samples_leaf=10, class_weight="balanced", random_state=42))
    ]),
    "XGBoost": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", xgb.XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.05, subsample=0.8, eval_metric="logloss", random_state=42))
    ])
}

print("=== Validation Set Model Comparison ===")
for name, pipe in models.items():
    pipe.fit(X_train, y_train)
    val_pred = pipe.predict(X_val)
    f1 = f1_score(y_val, val_pred)
    rec = recall_score(y_val, val_pred)
    prec = precision_score(y_val, val_pred)
    auc = roc_auc_score(y_val, pipe.predict_proba(X_val)[:, 1])
    print(f"{name:22s} | F1: {f1:.4f} | Recall: {rec:.4f} | Precision: {prec:.4f} | AUC: {auc:.4f}")

## 7. Conclusions & Findings
- **Random Forest** achieved the highest validation F1 and Recall, effectively catching struggling students early.
- Prior scores (`previous_assessment_score`, `average_previous_score`) and attendance percentage are the most influential signals.
- The complete production pipeline is packaged in `src/train.py`, `src/evaluate.py`, and interactive inference in `app/streamlit_app.py`.